## Gerald grounder — free GPU fallback

In Runtime > Change runtime type, enable a T4 GPU. Then choose Run all. When the tunnel starts, copy the printed `MOONDREAM_ENDPOINT` and `MOONDREAM_API_KEY` values into Gerald's Grounder settings.

In [ ]:
!pip install "transformers<5" accelerate fastapi uvicorn pillow

In [ ]:
%%writefile /content/server.py
from __future__ import annotations

import base64
import binascii
import hmac
import io
import re
import secrets

import torch
from fastapi import FastAPI, Header, HTTPException
from PIL import Image, UnidentifiedImageError
from pydantic import BaseModel
from transformers import AutoModelForCausalLM

MODEL_ID = "vikhyatk/moondream2"
MODEL_REVISION = "2025-06-21"
TOKEN = secrets.token_urlsafe(16)
print(f"Moondream grounder token: {TOKEN}")

model = None
app = FastAPI()


class PointRequest(BaseModel):
    image_url: str
    object: str
    model: str | None = None


def load_model(device: str | None = None):
    target = device or ("cuda" if torch.cuda.is_available() else "cpu")
    if target == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available")
    dtype = torch.float16 if target == "cuda" else torch.float32
    loaded = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        trust_remote_code=True,
        torch_dtype=dtype,
    )
    return loaded.to(target).eval()


@app.on_event("startup")
async def load_model_on_startup() -> None:
    global model
    if model is None:
        model = load_model()


@app.get("/health")
async def health() -> dict[str, bool]:
    return {"ok": model is not None}


@app.post("/v1/point")
async def point(
    request: PointRequest,
    x_moondream_auth: str | None = Header(default=None),
) -> dict[str, object]:
    if not x_moondream_auth or not hmac.compare_digest(x_moondream_auth, TOKEN):
        raise HTTPException(status_code=401, detail="Invalid token")
    header, separator, encoded = request.image_url.partition(",")
    if not separator or not re.fullmatch(r"data:image/[A-Za-z0-9.+-]+;base64", header):
        raise HTTPException(status_code=400, detail="Invalid image_url")
    try:
        image_data = base64.b64decode(encoded, validate=True)
        with Image.open(io.BytesIO(image_data)) as opened:
            image = opened.convert("RGB")
    except (binascii.Error, UnidentifiedImageError, ValueError):
        raise HTTPException(status_code=400, detail="Invalid image_url") from None
    if model is None:
        raise HTTPException(status_code=503, detail="Model is not loaded")
    return {"points": model.point(image, request.object)["points"]}


def parse_trycloudflare_url(output: str) -> str | None:
    match = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", output)
    return match.group(0) if match else None


In [ ]:
import server

server.model = server.load_model(device="cuda")
app = server.app
TOKEN = server.TOKEN

In [ ]:
import os
import queue
import subprocess
import threading
import time
import urllib.request

import uvicorn

uvicorn_server = uvicorn.Server(
    uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
)
server_thread = threading.Thread(target=uvicorn_server.run, daemon=True)
server_thread.start()
for _ in range(120):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=1):
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Grounder server did not start")

cloudflared_path = "/usr/local/bin/cloudflared"
urllib.request.urlretrieve(
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    cloudflared_path,
)
os.chmod(cloudflared_path, 0o755)
tunnel = subprocess.Popen(
    [cloudflared_path, "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
lines = queue.Queue()

def collect_tunnel_output():
    for line in tunnel.stdout:
        lines.put(line)

threading.Thread(target=collect_tunnel_output, daemon=True).start()
deadline = time.monotonic() + 60
tunnel_url = None
while time.monotonic() < deadline and tunnel.poll() is None:
    try:
        line = lines.get(timeout=1)
    except queue.Empty:
        continue
    print(line.rstrip())
    tunnel_url = server.parse_trycloudflare_url(line)
    if tunnel_url:
        break
if not tunnel_url:
    raise RuntimeError("Could not find a trycloudflare.com URL in cloudflared output")
print("Paste these into Gerald's Grounder settings:")
print(f"MOONDREAM_ENDPOINT={tunnel_url}/v1")
print(f"MOONDREAM_API_KEY={TOKEN}")